In [1]:
# make autoreload cell
%load_ext autoreload
%autoreload 2



In [3]:
import pandas as pd
from pathlib import Path


In [7]:
resolutions = pd.read_parquet(Path("resolutions_flat.parquet"))

In [4]:
resolutions

,id,type,date,year,weekday,paragraph_texts,resolutions_text
0,session-3788-num-1-resolution-1,resolution,1733-01-02,1733,vrijdag,"[""ONtfangen een Missive van den Resident Spina...","ONtfangen een Missive van den Resident Spina, ..."
1,session-3788-num-1-resolution-10,resolution,1733-01-02,1733,vrijdag,"[""IS ter Vergaderinge gelesen de Requeste van ...",IS ter Vergaderinge gelesen de Requeste van de...
2,session-3788-num-1-resolution-11,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont en geexhibe...",17 Ynde ter Vergaderinge getoont en geexhibeer...
3,session-3788-num-1-resolution-12,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont ende geëxhi...",17 Ynde ter Vergaderinge getoont ende geëxhibe...
4,session-3788-num-1-resolution-13,resolution,1733-01-02,1733,vrijdag,"[""OP de Requeste van de gesamentlijcke Straatm...",OP de Requeste van de gesamentlijcke Straatmaa...
...,...,...,...,...,...,...,...
692151,session-3262-num-99-resolution-5,resolution,1656-05-09,1656,dinsdag,"[""Ontfangen een Missive vande Heeren Gedeputee...",Ontfangen een Missive vande Heeren Gedeputeerd...
692152,session-3262-num-99-resolution-6,resolution,1656-05-09,1656,dinsdag,"[""Ontfangen een Missive vanden Hoochschouttett...",Ontfangen een Missive vanden Hoochschouttetten...
692153,session-3262-num-99-resolution-7,resolution,1656-05-09,1656,dinsdag,"[""Ontfangen een Missive vanden drossaert Itter...",Ontfangen een Missive vanden drossaert Itterso...
692154,session-3262-num-99-resolution-8,resolution,1656-05-09,1656,dinsdag,"[""Is ter Vergaderinge gelesen seecker advis va...",Is ter Vergaderinge gelesen seecker advis vand...


In [8]:
from kwic_search import scan_text

In [7]:
?scan_text

Signature:
scan_text(
    text: 'str',
    source: 'str' = '',
    window_tokens: 'int' = 10,
    min_score: 'int' = 75,
) -> 'list[KWICRecord]'
Docstring:
Find HOE spans in a raw (un-annotated) text.

Uses the compiled kws_hoe regex rules from hoe_classify to locate candidate
spans, then classifies each match.  Only matches with classifier score >=
*min_score* are returned.

Parameters
----------
text : str
    Full text to scan (a letter, paragraph, etc.).
source : str
    Label stored in each KWICRecord.source (e.g. the letter filename).
window_tokens : int
    Number of whitespace-delimited tokens to include on each side of the
    span in the KWIC record.
min_score : int
    Minimum hoe_classify score to keep a hit (0–100).

Returns
-------
list[KWICRecord]
    One record per identified HOE span, with surrounding context.
File:      ~/develop/republic_ner_matching/kwic_search.py
Type:      function

In [9]:
s_res = resolutions.sample(500).paragraph_texts.apply(lambda texts: scan_text(texts, "resolutie", window_tokens=5))

In [51]:
s_res.iloc[0][0].left_tokens

['ende', 'van', 'wegen', 'de', 'gemeene']

In [4]:
# how to find a name in the KWICRecord? 
# lets first look at the structure of the KWICRecords
# but lets use the patterns from the patterns_reference dataframe first
pr = pd.read_parquet(Path("data/patterns_reference.parquet"))


In [18]:
def get_names_from_kwic_record(kwic_record, patterns_reference):
    
    return []  # return a list of names found in the kwic_record

,cons_id_str,pattern,year_min,year_max
0,18744,abbink,1757.0,1792.0
1,18744,abbinck,1757.0,1792.0
2,18744,ablinck,1757.0,1792.0
3,18379,ablaing,1747.0,1775.0
4,18379,d' ablaing,1747.0,1775.0
...,...,...,...,...
8078,republic_add_13,quint,1694.0,1743.0
8079,republic_add_13,nan,1694.0,1743.0
8080,republic_add_04,burmania,1721.0,1756.0
8081,republic_add_04,van burmania,1721.0,1756.0


In [5]:
# first build a pattern tfidf store from the patterns_reference dataframe
from sklearn.feature_extraction.text import TfidfVectorizer


def build_pattern_tfidf_store(patterns_reference):
    # Keep pattern strings intact; do not split into characters.
    texts = patterns_reference.pattern.astype(str).str.strip()
    # Drop placeholder and empty values that pollute nearest-neighbor matches.
    texts = texts[texts.str.lower().ne("nan") & texts.ne("")]
    texts = texts.tolist()

    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
    mat = vec.fit_transform(texts)
    return {
        "vec": vec,
        "mat": mat,
        "patterns": texts,
    }


def match_hoe_store(
    store: dict,
    text: str,
    top_k: int = 1,
) -> list[tuple[str, float]]:
    """Return top-k (pattern_text, cosine_score) for *text*.

    Scores are in [0, 1]; higher is better.
    """
    q = store["vec"].transform([text])
    scores = (store["mat"] @ q.T).toarray().ravel()
    top_idx = scores.argsort()[::-1][:top_k]
    return [
        (store["patterns"][i], float(scores[i]))
        for i in top_idx
    ]


def list_apply_match(kwic_records=None, patstore=None):
    result = []
    # Handle both single KWICRecord and list of KWICRecords
    if isinstance(kwic_records, list):
        for kwic_record in kwic_records:
            for context in [kwic_record.left_tokens, kwic_record.right_tokens]:
                for term in context:
                    result.extend(match_hoe_store(patstore, term))
    else:
        for context in [kwic_records.left_tokens, kwic_records.right_tokens]:
            for term in context:
                result.extend(match_hoe_store(patstore, term))
    return result


patstore = build_pattern_tfidf_store(pr)


In [6]:
patstore

{'vec': TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)),
 'mat': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 312136 stored elements and shape (7850, 17764)>,
 'patterns': ['abbink',
  'abbinck',
  'ablinck',
  'ablaing',
  "d' ablaing",
  "' ablaing - giessenburgh",
  "' ablaing van giessenburgh",
  "'abaing - giessenburgh",
  '0d ablaing- giessenburgh',
  "14 d'ablaing - giessenburgh",
  "3 d'ablaing - giessenburgh",
  "8' ablaing- giessenburgh",
  'ablaing - giessenburg',
  "bout d' ablaing van giessenburgh",
  "bout d' ablaing-giessenburgh",
  "bout d'ablaing van giessenburgh",
  'd ablaing van giesenburgh',
  'd ablaing van giessenburg',
  'd ablaing van giessenburgh',
  'dablaing van giessenburgh',
  'jd ablaing van giessenburgh',
  'p ablaing-giefsenburgh',
  "v' ahaing- giessenburgh",
  "v'abaing - giessenburgh",
  "v'ablaing - giessenburgh",
  "v'ablaing van giessenburgh",
  'vablaing van giessenburgh',
  'van palland tot glinthuys queysen',
  "w' abl

In [10]:
s_res.apply(lambda kwic_record: list_apply_match(kwic_record, patstore))


258647    [(tour, 0.32705282481530357), (quar les, 0.573...
330382    [(van der hem, 0.796428911742489), (taas van a...
582274    [(vanden bemden, 0.4514126498628879), (leenwen...
340010    [(van burmania, 0.0), (van heeckeren van brant...
682050    [(grissioen, 0.23135818966049393), (fabricius,...
                                ...                        
455616    [(bight, 0.5918012170943414), (daay, 0.4502388...
443624    [(van der hem, 0.796428911742489), (coehoorn, ...
229768    [(van burmania, 0.0), (van heeckeren van brant...
203992    [(van burmania, 0.0), (van westreenen, 0.48708...
150390    [(elas, 0.37897884077199295), (1720 tot voldoe...
Name: paragraph_texts, Length: 500, dtype: object